# **Problem Statement**

## **Business Context**

ShopNest Global is a large-scale e-commerce platform operating across 30+ countries, serving over 50 million active customers and employing 2,000+ human support agents who run 24/7 across the US, Europe, India, and Southeast Asia, across product categories including electronics, fashion, groceries, and home appliances.

ShopNest processes over 200,000 orders per day. With every order comes the possibility of a delivery delay, a payment failure, a wrong item, or a return request. Customers reach out to the support team through tickets to report these issues and expect a fast, accurate resolution. These tickets are written in highly unstructured ways:

- Some are overloaded with background details where the real issue is buried.
- Others use abbreviations, shorthand, and order codes that are hard to interpret.
- Some contain so little information that the issue is entirely unclear.

As a result:

- Human agents spend the first **2–3 minutes** on each ticket just decoding what the customer is asking before any resolution work can begin.
- During peak periods (sales, holidays, logistics disruptions), daily ticket volume can spike from ~5,000 to **~15,000**, multiplying this inefficiency.
- High volume and inconsistent ticket content contribute to agent fatigue and higher error rates when accuracy is most critical.
- Drafting responses is manual, slow, and inconsistent in tone and clarity across agents, leading to suboptimal customer experiences.

## **Objective**

The objective is to build a POC of an AI-powered ticket intelligence system for ShopNest Global that:

1. **Summarises** incoming raw, unstructured tickets into a clean, concise summary for the support agent.
2. **Evaluates** the generated summary using an LLM-as-Judge approach, scoring quality on defined criteria.
3. **Generates** a professional, empathetic customer response grounded in ShopNest's support policies.
4. **Evaluates** the generated response using an LLM-as-Judge approach, scoring resolution quality.
5. **Compiles** all outputs into a single structured table and exports it for downstream use.

The end goal is to demonstrate that AI-assisted summarisation and response generation can meaningfully improve the consistency and quality of customer support operations at scale.

## **Data Dictionary**

| Column Name         | Data Type | Description                                                       |
| ------------------- | --------- | ----------------------------------------------------------------- |
| support_ticket_id   | Integer   | Unique identifier assigned to each support ticket                 |
| support_ticket_text | String    | Free-form text describing the issue or request raised by the user |

# **Installing and Importing Necessary Libraries**

In [1]:
# Install LangChain and OpenAI for LLM API access, and pandas for data handling
# Pinned versions ensure reproducibility across environments
%pip install pandas==2.2.2 langchain-openai==1.1.12 #openai==2.31.0
# Install Google Generative AI SDK, pandas, and pydantic for schema verification
%pip install google-generativeai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.4 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# Core libraries
import os            # For environment variable access (API key fallback)
import json
import re            # For parsing structured evaluator output into numeric scores
import pandas as pd  # For loading, manipulating, and exporting tabular data
# from langchain_openai import ChatOpenAI  # OpenAI
# from openai import OpenAI
from dotenv import load_dotenv
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## **Loading the Open API Key**

In [ ]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

In [5]:
# # Load the JSON file and extract values
# file_name = 'config.json'                                                       # Name of the configuration file
# with open(file_name, 'r') as file:                                              # Open the config file in read mode
#     config = json.load(file)                                                    # Load the JSON content as a dictionary
#     OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
#     OPENAI_API_BASE = config.get("OPENAI_API_BASE")
# # Store API credentials in environment variables
# os.environ['OPENAI_API_KEY'] = os.get("OPENAI_API_KEY") #OPENAI_API_KEY                                  # Set API key as environment variable
# os.environ["OPENAI_BASE_URL"] = os.get("OPENAI_API_BASE") #OPENAI_API_BASE                                 # Set API base URL as environment variable
# client = OpenAI()

#import from .env file using python-dotenv
# Load environment variables from the .env file
#load_dotenv()
# Fetch the API key
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') #os.getenv("GOOGLE_API_KEY")

# Setup environment fallback for the API key
api_key = GOOGLE_API_KEY
if not api_key:
    raise ValueError("Error: GOOGLE_API_KEY not found in environment variables. Please set it before running the notebook.")

# **Data Loading**

## **Load the data**

In [ ]:
# Uncomment the lines below if the dataset is stored in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# Write your code here

# Read the source data using pandas
# Adjust 'support_tickets.csv' to match your actual file path if needed
try:
    df = pd.read_csv('support_ticket_data.csv')
    print(f"Successfully loaded {len(df)} support tickets.")
except FileNotFoundError:
    print("Error: 'support_tickets.csv' not found. Please verify your file path.")

Successfully loaded 30 support tickets.


## **Data Overview**

In [7]:
# Write your code here

# Preview structure and first few entries
print("--- First 3 Rows ---")
display(df.head(3))

# Quick diagnostic on columns, non-null counts, and data types
print("\n--- DataFrame Summary ---")
df.info()

--- First 3 Rows ---


,support_ticket_id,support_ticket_desc
0,1,I cannot believe the level of service I have r...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri..."
2,3,not working. please help.



--- DataFrame Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   support_ticket_id    30 non-null     int64 
 1   support_ticket_desc  30 non-null     object
dtypes: int64(1), object(1)
memory usage: 612.0+ bytes


# **Summarization**

## **Set up an LLM**

In [19]:
# Write your code here

# We use 'gemini-2.5-flash' as the primary model for optimal speed and text extraction efficiency
PRIMARY_MODEL = "gemini-2.5-flash-lite"
print(f"Gemini API configured successfully using model: {PRIMARY_MODEL}")

client = genai.configure(api_key=api_key)

Gemini API configured successfully using model: gemini-2.5-flash-lite


I chose Gemini 2.5 Flash because its free for me (I have an Edu account) and willdefinitely be smart enough to figure out the tasks like `Categorisation`.

## **Set parameters**

In [20]:
# Write your code here

# Configure operational constraints for text summarization
# Low temperature enforces factual, highly predictable extractions
summarize_config = genai.GenerationConfig(
    temperature=0.1,
    max_output_tokens=200
)

## **Prompting Technique**

### **System Message**

In [21]:
# Write your code here

# System instructions establish the strict persona and boundaries for the model execution
summ_system_instruction = (
    "You are an elite Operations Analyst at ShopNest Global.\n"
    "Your objective is to ingest noisy, poorly formatted customer support tickets and condense them.\n"
    "Extract and highlight:\n"
    "- The exact core problem (e.g., delivery delay, incorrect size, broken payment flow).\n"
    "- Crucial technical attributes like Order IDs, tracking numbers, or abbreviations.\n"
    "- Customer sentiment context.\n"
    "Output instructions: Keep your response as a clean, highly scannable bulleted summary. Avoid wordy prose."
)

### **Generate Ticket Summaries**

In [22]:
# Write your code here

def process_ticket_summary(ticket_text):
    """Wrapper function utilizing GenerativeModel with a strict execution architecture."""
    model = genai.GenerativeModel(
        model_name=PRIMARY_MODEL,
        generation_config=summarize_config,
        system_instruction=summ_system_instruction
    )
    try:
        response = model.generate_content(str(ticket_text))
        return response.text.strip()
    except Exception as e:
        return f"Pipeline Error during execution: {str(e)}"

# Apply the pipeline across the support_ticket_text column
print("Processing summarization loop across dataset...")
df['generated_summary'] = df['support_ticket_desc'].apply(process_ticket_summary)
print("Summarization complete.")

Processing summarization loop across dataset...


Summarization complete.


### **Observations**

In [23]:
df

,support_ticket_id,support_ticket_desc,generated_summary
0,1,I cannot believe the level of service I have r...,Here's a summary of the customer support ticke...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",* **Core Problem:** Payment failure at check...
2,3,not working. please help.,* **Core Problem:** Unspecified technical is...
3,4,Okay so I have genuinely had it with this comp...,Here's the condensed summary of the customer s...
4,5,refund not received,* **Core Problem:** Refund not received.\n* ...
5,6,Hi. I placed an order for a laptop. It is a De...,Here's a summary of the customer's support tic...
6,7,ORDER SNX-6612 DELIVERED WRONG ITEM. I ORDERED...,* **Core Problem:** Incorrect item delivered...
7,8,"Hello, good morning. I hope this message finds...",* **Core Problem:** Follow-up on a recent or...
8,9,"hey so my package still hasnt shown up, its be...",* **Core Problem:** Package marked as delive...
9,10,I want to be very clear about how disappointed...,Here's a summary of the customer support ticke...


Write your observations here

1. A summary was generated.
2. However, Summary Generated did not really help me in Categorising the Support Ticket.
3. For those `generated_summary` with the "*Pipeline Error during execution: 429 POST http...*" result, I don't know why my Edu account is not kicking-in.

Possible Solutions:
- Few-Shot Prompts.
- High Input Noise-to-Signal Ratio (Wordy & Emotional Text)
1. Pattern: A significant portion of incoming tickets is filled with detailed backgrounds, emotional venting, customer loyalty histories, and explicit threats of escalation to third-party bodies (e.g., the Better Business Bureau, FTC, or consumer rights attorneys). The core operational issue is typically buried at the very end of the text.
1. Examples: * Ticket ID 1: Spends multiple sentences discussing loyalty since 2020, spent amounts, chatbot frustration, and BBB complaints before finally revealing the operational issue at the very end: Order SNX-4421, delivered wrong dishwasher model, wants replacement.
   - Ticket ID 4 & 24: Contain extensive narrative about "buck-passing" and repetitive historical carrier issues before arriving at the double charge or missing delivery problem.

-  Heavy Reliance on Shorthand, Acronyms, and Technical Jargon
1. Pattern: Power users or tech-savvy customers use highly dense, abbreviated language that requires specialized domain knowledge to understand quickly.
1. Examples:
  - Ticket ID 2: "Ord SNX-8902 ACH debit failed... CC also declined... Need NDD before the 15th... Pls check PG logs and escalate. Alt PMT: can store credit from prev RMA be applied..."
  - Ticket ID 23: "SNX-3301. DOA unit. Need RMA issued. Escalate to L2. Already past SLA window."

- Severely Low-Information (Vague) Tickets
1. Pattern: On the opposite end of the spectrum, some tickets contain almost zero operational metadata, context, order numbers, or specific product descriptions.
2. Examples:
  - Ticket ID 3: "not working. please help."
  - Ticket ID 5: "refund not received"

- Crucial Metadata Extraction
1. Pattern: For successful resolution, downstream fulfillment systems or human agents need specific keys: Order IDs, product names, tracking status, and explicit desired outcomes (e.g., replacement vs. refund).
1. AI Performance: The summarizer cleanly separates and formats these key tokens (e.g., isolating Order #SNX-9021, KitchenAid stand mixer, reverse double charge), ensuring that the resulting table contains structured, actionable data suitable for automated API triggers or CRM tags.


# **Evaluation for Summarization**

## **Setup an LLM**

In [24]:
# Write your code here

#LLM-as-Judge - Setup LLM for Eval by passing a strict Pydantic class map directly into Gemini's configuration. This guarantees structural compliance without brittle string extraction logic.

from pydantic import BaseModel, Field

# Declare a clean evaluation structure using Pydantic
class JudgeEvaluationSchema(BaseModel):
    score: int = Field(description="An integer rating strictly between 1 and 5.")
    reasoning: str = Field(description="A concise narrative detailing the justification for the given score.")

# Instantiate evaluation settings specifying JSON layout compliance
eval_config = genai.GenerationConfig(
    temperature=0.0,  # Absolute zero temperature provides deterministic scoring criteria
    response_mime_type="application/json",
    response_schema=JudgeEvaluationSchema
)

## **System Message**

In [25]:
# Write your code here

eval_summ_system_instruction = (
    "You are a strict Quality Assurance Judge verifying operational data transformations.\n"
    "Compare an original customer ticket against its generated summary.\n"
    "Rate completeness and accuracy on a scale from 1 to 5:\n"
    "- 1: Critical errors, total hallucination, or missed the entire problem statement.\n"
    "- 5: Exceptional extraction capturing every critical tracking code and core issue cleanly.\n"
    "You must output valid JSON matching the schema provided."
)

## **Generate Evaluation Scores**

In [26]:
# Write your code here

def run_summary_judge(row):
    """Executes evaluation across both text targets, parsing out structural json cleanly."""
    model = genai.GenerativeModel(
        model_name=PRIMARY_MODEL,
        generation_config=eval_config,
        system_instruction=eval_summ_system_instruction
    )

    evaluation_prompt = (
        f"Original Support Ticket:\n{row['support_ticket_desc']}\n\n"
        f"Generated Summary Evaluation Candidate:\n{row['generated_summary']}"
    )

    try:
        response = model.generate_content(evaluation_prompt)
        parsed_json = json.loads(response.text)
        return parsed_json.get("score"), parsed_json.get("reasoning")
    except Exception as e:
        return None, f"Judge validation failure: {str(e)}"

# Process evaluations sequentially
print("Running summary evaluation judge loop...")
judge_results = df.apply(run_summary_judge, axis=1)

# Populate DataFrame split output destinations
df['summary_eval_score'] = [res[0] for res in judge_results]
df['summary_eval_reasoning'] = [res[1] for res in judge_results]
print("Summary evaluations completed.")

Running summary evaluation judge loop...


Summary evaluations completed.


## **Observations**

In [27]:
df

,support_ticket_id,support_ticket_desc,generated_summary,summary_eval_score,summary_eval_reasoning
0,1,I cannot believe the level of service I have r...,Here's a summary of the customer support ticke...,5.0,The generated summary accurately captures all ...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",* **Core Problem:** Payment failure at check...,5.0,The summary accurately captures all critical i...
2,3,not working. please help.,* **Core Problem:** Unspecified technical is...,3.0,The summary accurately identifies that the cor...
3,4,Okay so I have genuinely had it with this comp...,Here's the condensed summary of the customer s...,5.0,The generated summary accurately captures the ...
4,5,refund not received,* **Core Problem:** Refund not received.\n* ...,3.0,The summary accurately captures the core probl...
5,6,Hi. I placed an order for a laptop. It is a De...,Here's a summary of the customer's support tic...,5.0,The summary accurately captures the core issue...
6,7,ORDER SNX-6612 DELIVERED WRONG ITEM. I ORDERED...,* **Core Problem:** Incorrect item delivered...,5.0,The summary accurately captures all key detail...
7,8,"Hello, good morning. I hope this message finds...",* **Core Problem:** Follow-up on a recent or...,5.0,The summary accurately captures the core probl...
8,9,"hey so my package still hasnt shown up, its be...",* **Core Problem:** Package marked as delive...,5.0,The summary accurately captures the core probl...
9,10,I want to be very clear about how disappointed...,Here's a summary of the customer support ticke...,5.0,The summary accurately captures the core probl...


Write your observations here

1. The `429 Status` occured again and I will ensure my Edu account is good after this.
1. **Successful Summaries and Evaluations:** For the tickets where both summarization and evaluation were successful (approximately the first 10 entries),
- the model generally performed well.
- Most summary_eval_score values are 5, indicating 'Exceptional extraction capturing every critical tracking code and core issue cleanly'.
- Some scores are 3, with reasoning suggesting that while the core problem was identified, the summaries were brief or could have included more specific details, especially for vague tickets.

# **Response Generation**

## **Set up an LLM**

In [38]:
# Write your code here

# Utilizing the same robust core model definition
RESPONSE_MODEL = "gemini-3-flash-preview"

## **Set parameters**

In [39]:
# Write your code here

# Configure operational constraints for response generation
# A moderate temperature allows for some creativity while maintaining relevance.
# Set a reasonable max_output_tokens to control response length.
response_config = genai.GenerationConfig(
    temperature=0.7, # temperature=0.4, - We use a marginally higher temperature to produce more fluid, empathetic sentence structuring
    max_output_tokens=350, #500
)

## **Prompting Technique**

### **System Message**

In [40]:
# Write your code here

response_system_instruction = (
    "You are an empathetic, highly professional Senior Customer Support Specialist at ShopNest Global.\n"
    "Draft an authentic, helpful direct response to the customer based on their support issue.\n"
    "Operational Guardrails:\n"
    "- Acknowledge the core frustration gracefully and prioritize tracking down issues.\n"
    "- Align solutions with general platform policies (e.g., immediate internal escalations for verified payment drops, processing refund/re-shipment options if parcels are physically missing/damaged).\n"
    "- Maintain a warm, clear, and reassuring tone. Do not make open-ended promises you can't explicitly track or resolve right now."
)

### **Generate Ticket Summaries**

In [41]:
# Write your code here

def process_customer_response(row):
    """Generates customer draft copy combining initial context with the engineered brief."""
    model = genai.GenerativeModel(
        model_name=RESPONSE_MODEL,
        generation_config=response_config,
        system_instruction=response_system_instruction
    )

    generation_prompt = (
        f"Customer Ticket Text:\n{row['support_ticket_desc']}\n\n"
        f"Condensed Operational Digest:\n{row['generated_summary']}"
    )

    try:
        response = model.generate_content(generation_prompt)
        return response.text.strip()
    except Exception as e:
        return f"Pipeline Error during customer generation phase: {str(e)}"

print("Processing final draft response loops...")
df['generated_response'] = df.apply(process_customer_response, axis=1)
print("Customer drafting phase complete.")


Processing final draft response loops...


Customer drafting phase complete.


### **Observations**

In [42]:
df

,support_ticket_id,support_ticket_desc,generated_summary,summary_eval_score,summary_eval_reasoning,generated_response
0,1,I cannot believe the level of service I have r...,Here's a summary of the customer support ticke...,5.0,The generated summary accurately captures all ...,Subject: Urgent: Resolution for your Bosch Dis...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",* **Core Problem:** Payment failure at check...,5.0,The summary accurately captures all critical i...,Subject: Urgent Assistance: Payment Issues & S...
2,3,not working. please help.,* **Core Problem:** Unspecified technical is...,3.0,The summary accurately identifies that the cor...,Subject: We’re here to help: Let’s get this
3,4,Okay so I have genuinely had it with this comp...,Here's the condensed summary of the customer s...,5.0,The generated summary accurately captures the ...,Subject: Urgent: Resolution for Order #SNX-9
4,5,refund not received,* **Core Problem:** Refund not received.\n* ...,3.0,The summary accurately captures the core probl...,Subject: Update regarding your refund request ...
5,6,Hi. I placed an order for a laptop. It is a De...,Here's a summary of the customer's support tic...,5.0,The summary accurately captures the core issue...,Subject: Support for your Dell Laptop Order (O...
6,7,ORDER SNX-6612 DELIVERED WRONG ITEM. I ORDERED...,* **Core Problem:** Incorrect item delivered...,5.0,The summary accurately captures all key detail...,Pipeline Error during customer generation phas...
7,8,"Hello, good morning. I hope this message finds...",* **Core Problem:** Follow-up on a recent or...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...
8,9,"hey so my package still hasnt shown up, its be...",* **Core Problem:** Package marked as delive...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...
9,10,I want to be very clear about how disappointed...,Here's a summary of the customer support ticke...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...


Write your observations here

**Successful Responses were written**

# **Evaluation for Response Generation**

## **Setup an LLM**

In [43]:
# Write your code

# Re-using our structural Pydantic template for structural consistency
eval_resp_config = genai.GenerationConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=JudgeEvaluationSchema
)

## **System Message**

In [44]:
# Write your code here

eval_resp_system_instruction = (
    "You are a specialized Customer Experience Quality Judge at ShopNest Global.\n"
    "Critique the proposed agent response draft against the original ticket context.\n"
    "Evaluate on a 1-5 scale across Tone, Policy Alignment, Empathy, and Clear Next Steps:\n"
    "- 1: Tone is dismissive, completely misses fixing the issue, or hallucinated details.\n"
    "- 5: Exceptional resolution pathway, balanced empathy, actionable clear steps, and professional execution.\n"
    "Output must conform strictly to the specified JSON schema structure."
)

## **Generate Evaluation Scores**

In [47]:
# Write your code here

def run_response_judge(row):
    """Executes final automated QA scoring checks across generated response output drafts."""
    model = genai.GenerativeModel(
        model_name=RESPONSE_MODEL,
        generation_config=eval_resp_config,
        system_instruction=eval_resp_system_instruction
    )

    evaluation_prompt = (
        f"Original Support Ticket:\n{row['support_ticket_desc']}\n\n"
        f"Proposed Agent Response Draft:\n{row['generated_response']}"
    )

    try:
        response = model.generate_content(evaluation_prompt)
        parsed_json = json.loads(response.text)
        return parsed_json.get("score"), parsed_json.get("reasoning")
    except Exception as e:
        return None, f"Response Judge execution exception: {str(e)}"

print("Executing automated response verification checks...")
resp_judge_results = df.apply(run_response_judge, axis=1)

# Mapping targets out cleanly
df['response_eval_score'] = [res[0] for res in resp_judge_results]
df['response_eval_reasoning'] = [res[1] for res in resp_judge_results]
print("All analytical judgment routines finalized.")

Executing automated response verification checks...


All analytical judgment routines finalized.


## **Observations**

In [48]:
df

,support_ticket_id,support_ticket_desc,generated_summary,summary_eval_score,summary_eval_reasoning,generated_response,response_eval_score,response_eval_reasoning
0,1,I cannot believe the level of service I have r...,Here's a summary of the customer support ticke...,5.0,The generated summary accurately captures all ...,Subject: Urgent: Resolution for your Bosch Dis...,1.0,The proposed response is critically incomplete...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",* **Core Problem:** Payment failure at check...,5.0,The summary accurately captures all critical i...,Subject: Urgent Assistance: Payment Issues & S...,1.0,"The proposed response is incomplete, consistin..."
2,3,not working. please help.,* **Core Problem:** Unspecified technical is...,3.0,The summary accurately identifies that the cor...,Subject: We’re here to help: Let’s get this,1.0,"The proposed response is incomplete, consistin..."
3,4,Okay so I have genuinely had it with this comp...,Here's the condensed summary of the customer s...,5.0,The generated summary accurately captures the ...,Subject: Urgent: Resolution for Order #SNX-9,1.0,The proposed response is incomplete and consis...
4,5,refund not received,* **Core Problem:** Refund not received.\n* ...,3.0,The summary accurately captures the core probl...,Subject: Update regarding your refund request ...,1.0,The proposed response is completely inadequate...
5,6,Hi. I placed an order for a laptop. It is a De...,Here's a summary of the customer's support tic...,5.0,The summary accurately captures the core issue...,Subject: Support for your Dell Laptop Order (O...,1.0,The proposed response is incomplete and cuts o...
6,7,ORDER SNX-6612 DELIVERED WRONG ITEM. I ORDERED...,* **Core Problem:** Incorrect item delivered...,5.0,The summary accurately captures all key detail...,Pipeline Error during customer generation phas...,1.0,The proposed response is a technical system er...
7,8,"Hello, good morning. I hope this message finds...",* **Core Problem:** Follow-up on a recent or...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...,1.0,The proposed response is a raw technical error...
8,9,"hey so my package still hasnt shown up, its be...",* **Core Problem:** Package marked as delive...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...,1.0,The proposed response is a raw technical syste...
9,10,I want to be very clear about how disappointed...,Here's a summary of the customer support ticke...,5.0,The summary accurately captures the core probl...,Pipeline Error during customer generation phas...,1.0,The proposed response is a technical system er...


Write your observations here

# **Output & Compilation**

In [49]:
# Write your code here

# Reorder and filter output columns to align directly with upstream system requirements
consolidated_columns = [
    'support_ticket_id',
    'support_ticket_desc',
    'generated_summary',
    'summary_eval_score',
    'summary_eval_reasoning',
    'generated_response',
    'response_eval_score',
    'response_eval_reasoning'
]

## **Combine All Outputs into a Single Consolidated Table**

In [50]:
# Write your code here

final_export_df = df[consolidated_columns]

# **Business Insights & Recommendations**

In [51]:
# Print metrics to double check your data run before committing file export
print(f"Dataframe compilation complete. Validated Dimensions: {final_export_df.shape}")
print(f"Mean Summary Quality Score: {final_export_df['summary_eval_score'].mean():.2f}/5.00")
print(f"Mean Response Quality Score: {final_export_df['response_eval_score'].mean():.2f}/5.00")

# Render a final processing snapshot to the user workspace
display(final_export_df.head(2))

# Save output flat file down to disk for engineering downstream use cases
final_export_df.to_csv('shopnest_ticket_intelligence_compiled.csv', index=False)
print("File system write succeeded: 'shopnest_ticket_intelligence_compiled.csv'")

Dataframe compilation complete. Validated Dimensions: (30, 8)
Mean Summary Quality Score: 4.60/5.00
Mean Response Quality Score: 1.00/5.00


,support_ticket_id,support_ticket_desc,generated_summary,summary_eval_score,summary_eval_reasoning,generated_response,response_eval_score,response_eval_reasoning
0,1,I cannot believe the level of service I have r...,Here's a summary of the customer support ticke...,5.0,The generated summary accurately captures all ...,Subject: Urgent: Resolution for your Bosch Dis...,1.0,The proposed response is critically incomplete...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",* **Core Problem:** Payment failure at check...,5.0,The summary accurately captures all critical i...,Subject: Urgent Assistance: Payment Issues & S...,1.0,"The proposed response is incomplete, consistin..."


File system write succeeded: 'shopnest_ticket_intelligence_compiled.csv'


## **Business Insights**

Write your insights here

## **Recommendations**

Write your recommendations here

<font size=6>Power Ahead!</font>
___